# 8 — Attention vs Adjacency: formal weight-matrix comparison

Closes the §5c gap in [04_interpretability_plan.md](../notes/04_interpretability_plan.md). Quantifies how 4c GAT learned attention differs from the Pearson adjacency it is masked by.

Sections:
1. Setup & sanity (shared mask)
2. Per-window weight-matrix correlation (Pearson & Spearman)
3. Per-stock row-wise divergence (KL/JS/cosine/top-1)
4. Attention asymmetry — GAT's real degree of freedom over GCN
5. Per-edge divergence and sector-pair heatmap
6. Summary table (global + VIX stable/stress split)

**Caveat**: extracted attention weights may differ from training-time due to a known score-scaling bug in `extract_attention_weights_v2`.


## 0. Setup


In [ ]:
import os, sys
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/repo'):
        get_ipython().system('git clone https://github.com/adam-909/4yp.git /content/repo')
    else:
        get_ipython().system('cd /content/repo && git pull')
    os.chdir('/content/repo/4YP-main')
    RESULTS_BASE = "/content/drive/MyDrive/FINAL_RESULTS"
    from google.colab import drive
    drive.mount('/content/drive')
elif os.path.exists("/mnt/g/My Drive"):
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "/mnt/g/My Drive/FINAL_RESULTS"
else:
    os.chdir('/home/adam/new4YP/4YP-main')
    RESULTS_BASE = "FINAL_RESULTS"
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd(), "| RESULTS_BASE:", RESULTS_BASE)

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from scipy.spatial.distance import jensenshannon
import warnings; warnings.filterwarnings("ignore", category=FutureWarning)

from gml.experiment_utils import load_experiment_results
from settings.default import ALL_TICKERS, BBG_SECTORS

plt.rcParams.update({"font.size": 11, "figure.dpi": 120,
                     "axes.grid": True, "grid.alpha": 0.3, "figure.facecolor": "white"})

In [ ]:
GCN_EXPERIMENT = "3_GCN_rolling_pearson"; GCN_CONFIG = "lb20_th0.4_equity"; GCN_SEED = 42
GAT_EXPERIMENT = "4c_GATv2_rolling_pearson_th0.4_scFalse_resTrue"; GAT_CONFIG = "lb20_th0.4_equity"; GAT_SEED = 40

gcn = load_experiment_results(GCN_EXPERIMENT, GCN_SEED, config_name=GCN_CONFIG, base_dir=RESULTS_BASE)
gat = load_experiment_results(GAT_EXPERIMENT, GAT_SEED, config_name=GAT_CONFIG, base_dir=RESULTS_BASE)

pearson   = gcn["adjacency"]                      # (W, 88, 88) rolling Pearson weights (with mask applied)
gat_attn  = gat["attention_weights"].mean(axis=1) # (W, 88, 88) head-averaged
dates     = pd.to_datetime(gat["test_dates"])
tickers   = list(gcn.get("tickers", ALL_TICKERS))[:gat_attn.shape[-1]]
sectors   = pd.Series(BBG_SECTORS).reindex(tickers)

mask = (pearson != 0)  # shared Pearson mask
print("pearson:", pearson.shape, "| gat_attn:", gat_attn.shape)
print("mean edges per window:", mask.sum(axis=(1,2)).mean())

In [ ]:
OUT_DIR_8 = "notebooks/8_results"
os.makedirs(OUT_DIR_8, exist_ok=True)

## 8.1 Mask sanity


In [ ]:
# 8.1: Sanity — do GAT and Pearson share the same mask?
gat_nz = (gat_attn > 1e-12)
agree  = (gat_nz == mask).mean(axis=(1,2))
print(f"mask agreement per window: mean {agree.mean():.4f}, min {agree.min():.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12,3))
ax[0].plot(dates, mask.sum(axis=(1,2)), label="Pearson edges"); ax[0].plot(dates, gat_nz.sum(axis=(1,2)), "--", alpha=0.6, label="GAT nonzero")
ax[0].set_title("Edge count per window"); ax[0].legend(); ax[0].tick_params(axis="x", rotation=30)
ax[1].hist(agree, bins=40); ax[1].set_title("Per-window mask agreement")
plt.tight_layout(); plt.show()

## 8.2 Per-window weight-matrix correlation
Computed only over edges present in the shared Pearson mask at each window — preserves time.


In [ ]:
# 8.2: Per-window weight-matrix correlation (Pearson & Spearman) over mask-present edges
W = gat_attn.shape[0]
rho_p = np.full(W, np.nan); rho_s = np.full(W, np.nan)
for t in range(W):
    m = mask[t]
    if m.sum() < 10: continue
    a = gat_attn[t][m]; b = pearson[t][m]
    rho_p[t] = stats.pearsonr(a, b)[0]
    rho_s[t] = stats.spearmanr(a, b)[0]

corr_df = pd.DataFrame({"pearson_rho": rho_p, "spearman_rho": rho_s}, index=dates)
print(corr_df.describe())

fig, ax = plt.subplots(figsize=(14,4))
ax.plot(dates, rho_p, label=f"Pearson ρ (mean {np.nanmean(rho_p):.3f})")
ax.plot(dates, rho_s, label=f"Spearman ρ (mean {np.nanmean(rho_s):.3f})", alpha=0.7)
ax.axhline(0, color="k", lw=0.5); ax.set_title("Per-window correlation: GAT attention vs Pearson weights (within mask)")
ax.legend(); ax.tick_params(axis="x", rotation=30); plt.tight_layout(); plt.show()

corr_df.to_csv(f"{OUT_DIR_8}/per_window_weight_correlation.csv")

## 8.3 Per-stock row-wise divergence
For each (stock, window), compare the row of learned attention to the row-normalised Pearson adjacency.


In [ ]:
# 8.3: Per-stock row-wise distributional comparison (KL, JS, cosine, top-1 agreement)
eps = 1e-12
# Normalise Pearson rows over mask (abs values → softmax-style distribution)
pearson_abs = np.abs(pearson)
row_sum = pearson_abs.sum(axis=2, keepdims=True) + eps
pearson_p = pearson_abs / row_sum
gat_p = gat_attn / (gat_attn.sum(axis=2, keepdims=True) + eps)

W, N, _ = gat_attn.shape
js  = np.full((W, N), np.nan)
cos = np.full((W, N), np.nan)
top1 = np.full((W, N), np.nan)
for t in range(W):
    for i in range(N):
        if mask[t, i].sum() < 2: continue
        p = pearson_p[t, i]; q = gat_p[t, i]
        js[t, i]  = jensenshannon(p, q)   # sqrt of JS divergence; 0 = identical
        cos[t, i] = (p @ q) / (np.linalg.norm(p)*np.linalg.norm(q) + eps)
        top1[t, i] = float(np.argmax(p) == np.argmax(q))

per_stock = pd.DataFrame({
    "ticker": tickers,
    "sector": sectors.values,
    "mean_JS":   np.nanmean(js,   axis=0),
    "mean_cos":  np.nanmean(cos,  axis=0),
    "top1_rate": np.nanmean(top1, axis=0),
}).sort_values("mean_JS", ascending=False)
print("Top-10 stocks where GAT DISAGREES most with Pearson:")
print(per_stock.head(10).to_string(index=False))
print("\nBottom-10 (GAT ~ Pearson):")
print(per_stock.tail(10).to_string(index=False))

per_stock.to_csv(f"{OUT_DIR_8}/per_stock_row_divergence.csv", index=False)

# Per-sector aggregation
per_sector = per_stock.groupby("sector").agg(
    mean_JS=("mean_JS","mean"), mean_cos=("mean_cos","mean"),
    top1_rate=("top1_rate","mean"), n=("ticker","size")).sort_values("mean_JS", ascending=False)
print("\nPer-sector row divergence:")
print(per_sector)
per_sector.to_csv(f"{OUT_DIR_8}/per_sector_row_divergence.csv")

## 8.4 Asymmetry
`mean |A[i,j] - A[j,i]|` — GAT's unique degree of freedom (Pearson is symmetric by construction).


In [ ]:
# 8.4: Attention asymmetry — |A_ij - A_ji|  (Pearson is symmetric ⇒ zero)
iu = np.triu_indices(N, k=1)
asym_abs   = np.full(W, np.nan)
asym_norm  = np.full(W, np.nan)
for t in range(W):
    m = mask[t] & mask[t].T       # only pairs that are bidirectionally connected
    if m[iu].sum() < 5: continue
    A = gat_attn[t]
    diff = np.abs(A - A.T)
    summ = A + A.T + 1e-12
    pairs = m[iu]
    asym_abs[t]  = diff[iu][pairs].mean()
    asym_norm[t] = (diff[iu][pairs] / summ[iu][pairs]).mean()

asym_df = pd.DataFrame({"asym_abs": asym_abs, "asym_norm": asym_norm}, index=dates)
print(asym_df.describe())

fig, ax = plt.subplots(figsize=(14,3.5))
ax.plot(dates, asym_abs, label="mean |A_ij - A_ji|")
ax.set_title("GAT attention asymmetry over time (Pearson baseline = 0)"); ax.legend()
ax.tick_params(axis="x", rotation=30); plt.tight_layout(); plt.show()
asym_df.to_csv(f"{OUT_DIR_8}/asymmetry_timeseries.csv")

# Top-k persistent asymmetric pairs
A_mean = gat_attn.mean(axis=0)
diff_mean = np.abs(A_mean - A_mean.T)
pair_mask = mask.any(axis=0) & mask.any(axis=0).T
pairs = []
for i,j in zip(*iu):
    if pair_mask[i,j]:
        pairs.append((tickers[i], tickers[j], sectors.iloc[i], sectors.iloc[j],
                      diff_mean[i,j], A_mean[i,j], A_mean[j,i]))
pairs_df = pd.DataFrame(pairs, columns=["i","j","sec_i","sec_j","mean_abs_diff","A_ij","A_ji"])
pairs_df = pairs_df.sort_values("mean_abs_diff", ascending=False)
print("\nTop-20 most persistently asymmetric pairs:")
print(pairs_df.head(20).to_string(index=False))
pairs_df.to_csv(f"{OUT_DIR_8}/top_asymmetric_pairs.csv", index=False)

# Intra vs inter-sector asymmetry
pairs_df["intra"] = pairs_df["sec_i"] == pairs_df["sec_j"]
print("\nAsymmetry intra vs inter sector:")
print(pairs_df.groupby("intra")["mean_abs_diff"].agg(["mean","median","count"]))

## 8.5 Per-edge divergence + sector-pair heatmap


In [ ]:
# 8.5: Per-edge divergence (GAT row-normalised vs Pearson row-normalised)
div_edge = gat_p - pearson_p                 # (W, N, N)
mean_abs_div = np.nanmean(np.abs(div_edge), axis=0)  # (N, N)

# Top-20 most divergent edges
rows = []
for i in range(N):
    for j in range(N):
        if mask.any(axis=0)[i,j] and i != j:
            rows.append((tickers[i], tickers[j], sectors.iloc[i], sectors.iloc[j],
                         mean_abs_div[i,j]))
edge_df = pd.DataFrame(rows, columns=["src","dst","sec_src","sec_dst","mean_abs_div"]).sort_values("mean_abs_div", ascending=False)
print("Top-20 most divergent edges:")
print(edge_df.head(20).to_string(index=False))
edge_df.to_csv(f"{OUT_DIR_8}/per_edge_divergence.csv", index=False)

# Sector-pair heatmap of signed mean divergence (GAT - Pearson)
signed_mean = np.nanmean(div_edge, axis=0)
sec_levels = sorted(sectors.dropna().unique())
heat = np.zeros((len(sec_levels), len(sec_levels)))
counts = np.zeros_like(heat)
for i in range(N):
    for j in range(N):
        if i == j or not mask.any(axis=0)[i,j]: continue
        si = sectors.iloc[i]; sj = sectors.iloc[j]
        if pd.isna(si) or pd.isna(sj): continue
        a = sec_levels.index(si); b = sec_levels.index(sj)
        heat[a,b] += signed_mean[i,j]; counts[a,b] += 1
heat = heat / np.where(counts == 0, 1, counts)

fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(heat, xticklabels=sec_levels, yticklabels=sec_levels,
            cmap="RdBu_r", center=0, ax=ax, cbar_kws={"label":"mean (GAT_p - Pearson_p)"})
ax.set_title("Sector-pair mean divergence: GAT up/down-weights vs Pearson")
plt.tight_layout(); plt.show()
pd.DataFrame(heat, index=sec_levels, columns=sec_levels).to_csv(f"{OUT_DIR_8}/sector_pair_divergence.csv")

## 8.6 Summary and regime split


In [ ]:
# 8.6: Summary — global, and split by VIX regime
import yfinance as yf
vix = yf.download("^VIX", start="2017-01-01", end="2023-09-01")["Close"].squeeze()
vix.index = pd.to_datetime(vix.index)
vix_w = vix.reindex(dates, method="ffill")
stress = vix_w > vix_w.quantile(0.75)

def agg(mask_t):
    return {
        "mean_pearson_rho": np.nanmean(rho_p[mask_t]),
        "mean_spearman_rho": np.nanmean(rho_s[mask_t]),
        "mean_asym_abs": np.nanmean(asym_abs[mask_t]),
        "mean_row_JS": np.nanmean(js[mask_t]),
        "mean_top1_agreement": np.nanmean(top1[mask_t]),
    }
summary = pd.DataFrame({
    "all":    agg(np.ones(W, bool)),
    "stable": agg(~stress.values),
    "stress": agg(stress.values),
})
print(summary.round(4))
summary.to_csv(f"{OUT_DIR_8}/summary_metrics.csv")

## Caveats


In [ ]:
# Caveats for the thesis write-up:
# 1. `extract_attention_weights_v2` in gml/graph_attention_v2.py does NOT apply the
#    1/sqrt(units) score scaling the model used at training time. The 4c run trained
#    with scale_scores=True, so the attention here is softmax-of-unscaled-scores —
#    sharper than the weights actually used for prediction. All metrics should be
#    reported as "extracted (pre-bugfix)".
# 2. Head averaging: we mean across the 4 GAT heads before comparison. Per-head
#    disagreement across stocks could be a follow-up if any aggregate finding is close
#    to zero.
print("Notebook 8 complete. Artifacts under", OUT_DIR_8)